### Initialization

In [0]:
# include the silver_utils
import sys
sys.path.append('/Workspace/Users/mommensabry@gmail.com/Medical_Insurance_Analytics/lakehouse/libs')
from silver_utils import *

### Read Bronze table

In [0]:
df = spark.table("medical_insurance.default.department")
display(df)

### Silver Transformation

In [0]:
df = (
    df.transform(lambda d: remove_duplicates(d, ["department_id"]))
      .transform(trim_all_string_columns)
      .transform(replace_null_strings)
      .transform(replace_null_numeric)
)
replace_blank_date(df)

# add zero to phone numbers
df = df.withColumn("manager_phone", concat(lit("0"), col("manager_phone")))
df = format_egypt_phone_numbers(df, ["manager_phone"])


# edit thr floor_number data type to int
df = df.withColumn("floor_number", col("floor_number").cast("int"))

display(df)

### write into silver table

In [0]:
df.write \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .format("delta") \
  .saveAsTable("medical_insurance.silver.department_silver")

In [0]:
df.display()